In [1]:
from pathlib import Path

import pandas as pd

RESULTS = Path("results")

In [ ]:
sse_results = pd.read_parquet(RESULTS / "sse_outputs/cluster_table.parquet")

In [3]:
degree = sse_results[[
    "in_degree", "out_degree",
    "in_strength", "out_strength"
]]

In [4]:
print(degree.describe().to_string())

           in_degree     out_degree    in_strength   out_strength
count  277154.000000  277154.000000  277154.000000  277154.000000
mean        0.424251       0.424251       0.581323       0.581323
std         0.839130       0.838622       2.979068       3.015566
min         0.000000       0.000000       0.000000       0.000000
25%         0.000000       0.000000       0.000000       0.000000
50%         0.000000       0.000000       0.000000       0.000000
75%         1.000000       1.000000       1.000000       1.000000
max       224.000000     178.000000     676.000000     496.000000


In [13]:
sse_results["burden_eligible"].unique()

array([True, None, False], dtype=object)

In [22]:
burden_eligible = sse_results[sse_results["burden_eligible"] == True]

In [23]:
print(burden_eligible[[
    'burst_score',
    'burst_score_null_z',
    'burst_score_upper_p',
    'burden_score',
    'burden_score_null_z',
    'burden_score_upper_p',
]].corr().to_string())

                      burst_score  burst_score_null_z  burst_score_upper_p  burden_score  burden_score_null_z  burden_score_upper_p
burst_score              1.000000            0.964554            -0.948200     -0.224931            -0.230611              0.229218
burst_score_null_z       0.964554            1.000000            -0.979720     -0.199122            -0.202088              0.200948
burst_score_upper_p     -0.948200           -0.979720             1.000000      0.211479             0.214117             -0.212739
burden_score            -0.224931           -0.199122             0.211479      1.000000             0.996944             -0.993344
burden_score_null_z     -0.230611           -0.202088             0.214117      0.996944             1.000000             -0.998611
burden_score_upper_p     0.229218            0.200948            -0.212739     -0.993344            -0.998611              1.000000


In [6]:
sse_results["candidate_tier"].unique().tolist()

['background_or_low_information',
 'size_ineligible',
 'possible_review',
 'high_priority_burden',
 'high_priority_burst',
 'high_priority_both_axes']

In [7]:
HP = [
    "high_priority_burden",
    "high_priority_burst",
    "high_priority_both_axes"
]

BG = [
    "background_or_low_information",
    "possible_review"
]

candidates = sse_results[sse_results["candidate_tier"].isin(HP)]
background = sse_results[sse_results["candidate_tier"].isin(BG)]


DEFAULT_MIXING_FEATURES = [
    "sex_entropy_z",
    "age_entropy_z",
    "simd_entropy_z",
    "datazone_entropy_z",
    "local_authority_entropy_z",
    "urban_rural_entropy_z",
    "health_board_entropy_z",
    "sex_entropy_obs",
    "age_entropy_obs",
    "simd_entropy_obs",
    "datazone_entropy_obs",
    "local_authority_entropy_obs",
    "urban_rural_entropy_obs",
    "health_board_entropy_obs",
]

MIXING_TERTILE_FEATURES = [f"{feature}_tertile" for feature in DEFAULT_MIXING_FEATURES]

MIXING_TERTILE_ORDER = [
    "more_homogeneous",
    "as_expected",
    "more_mixed",
]

In [8]:
candidates["cluster_size"].describe()

count    631.000000
mean      35.112520
std       60.433027
min        6.000000
25%       12.000000
50%       22.000000
75%       37.000000
max      860.000000
Name: cluster_size, dtype: float64

In [12]:
len(candidates), len(background)

(631, 8331)

In [9]:
background["cluster_size"].describe()

count    8331.000000
mean       15.606890
std        41.038549
min         6.000000
25%         7.000000
50%         9.000000
75%        14.000000
max      1695.000000
Name: cluster_size, dtype: float64

In [15]:
sex = pd.read_parquet(RESULTS / "tables/cluster_composition_sex.parquet")

In [17]:
sex["sse_status"].value_counts()

sse_status
background    8331
candidate      631
Name: count, dtype: int64